# Fractional differentiate

This notebook will cover exercise answer.

* Exercise 5.4
* Exercise 5.5

As we go along, there will be some explanations.

Stationarity is a key concept in time-series, by now the idea itself has been demostrated in previous notebooks (Feat Importance).

Most of the functions below can be found under research/Features

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
from numba import njit
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

from scipy.stats import jarque_bera

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint

p = print

#pls take note of version
#numba 0.49.1 #https://github.com/numba/numba/issues/4255
#numpy 1.17.3
#pandas 1.0.3

dollar = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

# For most part of the func we only use 'close'

close = dollar['close'].to_frame()

In [ ]:
#The same func can be found under research/Feature
@njit
def getWeights(d, size):
    w=[1.]
    for k in np.arange(1,size):
        w_ = -w[-1]/k*(d-k+1)
        w.append(w_)
    w=np.array(w[::-1]).reshape(-1,1)
    return w

def fracDiff(series, d, thres=.01):
    w=getWeights(d, series.shape[0])
    w_=np.cumsum(abs(w))
    w_/=w_[-1]
    skip = w_[w_>thres].shape[0]
    df={}
    for name in series.columns:
        seriesF,df_=series[[name]].fillna(method='ffill').dropna(),pd.Series(index=series.index, dtype=float)
        for iloc in range(skip,seriesF.shape[0]):
            loc=seriesF.index[iloc]
            if not np.isfinite(series.loc[loc,name]): continue
            df_[loc]=np.dot(w[-(iloc+1):,:].T,seriesF.loc[:loc])[0,0]
        df[name]=df_.copy(deep=True)
    df=pd.concat(df,axis=1)
    return df

@njit
def getWeights_FFD(d, thres):
    w,k=[1.],1
    while True:
        w_ = -w[-1]/k*(d-k+1)
        if abs(w_) < thres:
            break
        w.append(w_); k+=1
    return np.array(w[::-1]).reshape(-1,1)

# need to refactor to optimze if not will take forever if threshold too low
def fracDiff_FFD(series, d, thres=1e-2):
    w, df = getWeights_FFD(d, thres), {}
    width = len(w)-1
    for name in series.columns:
        seriesF,df_=series[[name]].fillna(method='ffill').dropna(),pd.Series(index=series.index, dtype=float)
        for iloc in range(width, seriesF.shape[0]):
            loc0, loc1 = seriesF.index[iloc - width], seriesF.index[iloc]
            if not np.isfinite(series.loc[loc1,name]): continue
            df_[loc1]=np.dot(w.T,seriesF.loc[loc0:loc1])[0,0]
        df[name]=df_.copy(deep=True)
    df=pd.concat(df,axis=1)
    return df

def min_value(data: pd.Series, func, thres = 0.01, pval_threshold: float = 0.05):
    d_domain = np.linspace(start = 0, 
                           stop = 2, 
                           num=100, 
                           endpoint=True, 
                           retstep=False, 
                           dtype=float)
    
    for d in d_domain:
        df1 = np.log(data).resample('1D').last() # pls note downcast to daily obs
        df2 = func(df1, d, thres = thres).dropna()
        p(df2)
        df2 = adfuller(df2.squeeze(), maxlag=1, regression='c', autolag=None)
        try:
            if df2[1] <= pval_threshold:
                return d
        except:
            p('Something is wrong! Most likely required d value more than 2!!')

In [ ]:
# Take optimal value to pass stationary test
# if you are keen.. adfuller can provide full test result.. go to statsmodel API documentation for more details
# When you run this min_value func from rs, you may encounter memory issue.
test_val = 0.05

log_price = close.apply(np.log)

# this func is the same as others as seen above except input has to be log by users, func will no long do it.
# you have more flexiblility to input log price series or non-log, as well as cumsum()
mv = rs.min_value(data = log_price,
                  FFD = True,
                  thres = 0.01,
                  pval_threshold = test_val,
                  num = 100,
                  num_threads = 21)

# Do not set autolag to 'AIC'
adf_pval = adfuller(fracDiff_FFD(log_price, d=0.141414).dropna().squeeze(), 
                    maxlag=1, 
                    regression='c', 
                    autolag=None)[1]

p("Min d Value: {0:.6f}\nADF pVal: {1:.5f} with critical value: {2}%".format(mv, adf_pval, test_val * 100))

In [ ]:
ffd0 = fracDiff_FFD(log_price, d = 0.141)

In [ ]:
ffd0.describe() # initial num count 24079 before FFD

In [ ]:
ffd1 = fracDiff_FFD(ffd0.dropna(), d = -0.141)

In [ ]:
ffd1.describe() # after second FFD based on first FFD but with -d value

#### Further investgation on negative d value

Even without using FFD twice, just by using a single FFD with -d value. It will still produce NaNs

In [ ]:
ffd2 = fracDiff_FFD(ffd1, d = -0.141)

In [ ]:
ffd2.describe()

## Infinite observations

When we try to get weight for FFD, using positive d value will produce appropriate "discount" to observation values.

However, using negative d value we will instead get unlimited "additive" observation values, since we can never get to hit threshold.

In short, convergence will not occur but instead divergence will take place.

#### getWeights_FFD function warning

The function getWeight_FFD would have cause infinite loop when we use negative d value if we did not put a threshold limit condition.

In [ ]:
getWeights_FFD(-0.141, 0.01) # negative value which led to divergence (right to left)

In [ ]:
getWeights_FFD(0.141, 0.01) # positive value for convergence (left to right)

#### Exercise 5.5

In [ ]:
log_price.head(6)

In [ ]:
cumsum_logp = log_price.cumsum()
cumsum_logp.dropna(inplace = True)
cumsum_logp.describe() #initial count 24079 if no NaNs count should be same

In [ ]:
cumsum_logp.head(6)

In [ ]:
cumsum_logp.plot(figsize=(10,8)) # literally a straight line

**Note**

Cumumlative sum prices use threshold 1.e-5.

Non-cumumlative prices use threshold 1.e-2.

Otherwise threshold is too low.

Log prices in theory may improve convergence, you may wish to check max likelihood estimation or AIC score.

In [ ]:
# this is a sample func on per tick, DO NOT use it otherwise it will take forever if per tick.

def minVal_cs(data: pd.Series, thres: float = 1e-5, pval_threshold: float = 0.05, d_range: list = [1.9, 2.0] ):
    d_domain = np.linspace(start = d_range[0], 
                           stop = d_range[1], 
                           num=1e9, 
                           endpoint=True, 
                           retstep=False, 
                           dtype=float)
    
    for d in d_domain:
        try:
            df1 = np.log(data).cumsum() #.resample('1h').last()# pls note downcast to daily obs
            df1.dropna(inplace=True)
            df2 = fracDiff_FFD(df1, d, thres = thres).dropna()
            df2 = adfuller(df2.squeeze(), maxlag = 1,regression='c', autolag=None)
            if df2[1] <= pval_threshold:
                print(d)
                return d
        except:
            p('Something is wrong! Most likely required d value beyond input parameter!!')

In [ ]:
# Kindly refer to the ans d = 1.99999889 instead to save time, it may even crash

#even with multiprocessing running this func will still take some time
"""
mv = rs.min_value(data = cumsum_logp,
                  FFD = True,
                  thres = 1e-5,
                  pval_threshold = test_val,
                  num = 1e9,
                  num_threads = 21)
"""

minVal_cs(data = cumsum_logp, thres = 1e-5, pval_threshold = 0.05, d_range = [1.9999, 2.0])

**Note**

If you cumulative sum your financial time-series, before FFD. 

You will need to differentiate and in this case min d value is very close 2.0

In [ ]:
# if d value was 1.999999 ADF p value would be 0.01695 < 0.04165 < 0.05
ffd3 = fracDiff_FFD(cumsum_logp, 
                    d = 1.99999889, 
                    thres=1e-5
                   ).dropna()

adf_pval = adfuller(ffd3.squeeze(), 
                    maxlag = 1,
                    regression='c', 
                    autolag=None)[1]

p("\nADF pVal: {0:.5f} with critical value: {1}%".format(adf_pval, test_val * 100))

In [ ]:
fracdiff_series = pd.DataFrame(index=ffd3.index).assign(ffd3 = ffd3, #after fractional differentiate
                                                        cumsum_logp = cumsum_logp, #cumulative sum of log price
                                                        close = close) #original

fracdiff_series[['ffd3', 'close']].plot(secondary_y='close', figsize=(10,8)) #not even remotely close

In [ ]:
fracdiff_series[['ffd3', 'cumsum_logp']].plot(secondary_y='cumsum_logp', figsize=(10,8)) #not even remotely close

fracdiff_series.corr(method='pearson') # see correlation matrix

In [ ]:
p("ADF pVal for original time-series: {0}\n".format(adfuller(fracdiff_series['close'], 
                                  maxlag = 1, 
                                  regression = 'c', 
                                  autolag=None)[1]))

p("ADF pVal for acummulative log price time-series: {0}\n".format(adfuller(fracdiff_series['cumsum_logp'], 
                                  maxlag = 1, 
                                  regression = 'c', 
                                  autolag=None)[1]))

# time-series is not stationary since p-value is more than 0.05

p("Jarque Bera pVal: {0:.5f}\n".format(jarque_bera(fracdiff_series['ffd3'])[1]))

# time-series is not normal since p value is less than 0.05

result = coint(fracdiff_series['ffd3'], fracdiff_series['close'], maxlag = 1, trend = 'c', autolag = None)

p("Engel-Granger Coint pVal: {0}".format(result[1]))

# there is a long term relationship since p-value is less than 0.05

### Memory Persistance

ADF p-val for original was 0.2939. Corr against FFD was -0.405165, inverse relation.

ADF p-val for cumsum log price was 0.9979. Corr against FFD was -0.9989, inverse relation.

ADF p-val for FFD using d value 1.99999889 was 0.04165.(Stationary).

Coint p-val for FFD and original was 0.009683.

Initially we preserve time series memory by log price, however as we cumsum the log price time series we cause these memory to accumulate which increases ADF p-value and more "unstationary" since we created a linear trend.

In most cases, d value was less than 1 but due to this new cumsum log price we end up using above 1 which will cause "decay" to be very aggressive which is reflected in the graph ffd3 vs original time series.

However, when perform Engel-Granger Coint test, test result states the two seemingly negatively correlated time-series do have a long term relationship. Most likely due to the memory persistance, which achieved p-value of 0.009683 lower than critical size 0.01.

>"Log prices have memory but are non-stationary. Cointegration is the trick that makes regression works on non-stationary series, so that memory is preserved."
>
> Advances in Financial Machine Learning, page 88, section 5.7

### Conclusion

This is an important finding since we require a stationary series with maximum memory preserved to ensure machine learning use it as key predictive feature effectively.